# Sequence Likelihood

Score observed recall events under a memory-search model.

`jaxcmr.loss.sequence_likelihood` contains one implementation. It converts
recorded study positions to within-list item codes, creates the trial's model,
simulates study, enters retrieval, and computes probabilities along the observed
history. It then masks unscored probabilities and sums negative logarithms.

Every supplied response updates the model, including excluded first recalls.
A scoring mask changes the objective, not the conditioning history. The model's
item probabilities still include its stopping rule when STOP scores are excluded.

| Loss class | Scored events |
|---|---|
| `MemorySearchLikelihoodLoss` | Every supplied event, unless a custom mask is given. |
| `ExcludeFirstRecallLikelihoodLoss` | Slots after the first recall, including recorded STOP events. |
| `ExcludeTerminationLikelihoodLoss` | Nonzero item responses. |
| `ExcludeFirstRecallAndTerminationLikelihoodLoss` | Nonzero responses after the first slot. |

Choose the observation model for your experiment. Excluding an event does not
make the resulting objective equivalent to a likelihood conditional on that event.

```python
from jaxcmr.loss.sequence_likelihood import ExcludeTerminationLikelihoodLoss

loss = ExcludeTerminationLikelihoodLoss(model_factory_cls, data, features)
values = loss(trial_indices, base_params, free_param_names, candidate_parameters)
```

`trial_indices` contains selected dataset row indices. Candidate parameters have
shape `(parameters, candidates)` in physical units; the result has one loss per
candidate. The same loss constructor can be passed as a fitter's `loss_fn_cls`.

A custom `LikelihoodMaskFn` receives the converted one-dimensional recall
sequence and returns one Boolean per event:

```python
from functools import partial
import jax.numpy as jnp
from jaxcmr.loss.sequence_likelihood import MemorySearchLikelihoodLoss

def first_five_items(recalls):
    return (jnp.arange(recalls.size) < 5) & (recalls != 0)

loss_factory = partial(MemorySearchLikelihoodLoss, mask_likelihoods=first_five_items)
```

Impossible unscored events are replaced with probability one **before** taking
logarithms. Impossible scored events give infinite loss. The loss adds no epsilon
and does not clip probabilities; any probability floor belongs to the specified
model. JAX compilation and differentiation follow the model's numerical behavior.

Zero-coded slots represent STOP/padding in the recorded sequence. The default
loss scores every supplied slot, so select a mask that matches the recording
convention when multiple zero slots follow the recorded stop. Omitting a STOP
slot from a full recall row does not automatically add one.

For an unordered complete sequence with one terminal STOP, use the separate
[experimental permutation loss](permutation.ipynb). For behavioral summary
matching, see [MSE](mse.ipynb). [API reference](../../../guide/reference/loss-sequence.qmd).
